# 06 – Strategie-Vergleich (mehrere Strategien, ein Zeitraum)

Läuft **mehrere Strategien** (Default: `pairwise_zscore` vs. `garch_wilks`) nacheinander mit
denselben Baskets/Zeitraum/Basisparametern, speichert jeden Lauf einzeln via `src.backtest.runs.save_run`
und vergleicht die gespeicherten Runs (Kennzahlen, Equity-Kurven, Trade-Verteilung nach Exit-Grund) --
funktional das Pendant zu `notebooks/03_Tradeing/strategy_comparison.ipynb`, hier aber auf der
aktuellen Engine (`src/backtest/`) statt der alten, separaten `pairtrading_engine.py`/
`garch_pairtrading_engine.py`/`bollinger_pairtrading_engine.py`.

`garch_wilks` (`GarchWilksToleranceStrategy`, siehe `src/backtest/strategy.py`) portiert die
GARCH(1,1)/Wilks-Toleranzintervall-Signalkette (Lin, Chen & Syu 2021) aus der alten
`garch_pairtrading_engine.py` auf die aktuelle Engine: Entry, sobald der GARCH-normierte
Return-Spread `Xt` des extremsten Paares einer Clique eine rollierend geschätzte obere
Wilks-Toleranzgrenze durchbricht; Exit, sobald `|Xt|` des gehaltenen Paares die Mitte zwischen
unterer und oberer Grenze erreicht -- eine strukturell andere Entry/Exit-Logik als `pairwise_zscore`
(fester `entry_z`/`exit_z`-Schwellenwert auf einem bei Formation eingefrorenen Preis-Spread-Z-Score).

**Nicht direkt ausführen.** `cp _template.ipynb`-Konvention gilt auch hier: beim Kopieren nur
Abschnitt 1 (Konfiguration) und Abschnitt 10 (Interpretation) anpassen. Siehe `README.md` in
diesem Ordner für die volle Konvention.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("../..").resolve()))

import time
import pandas as pd
from IPython.display import display

from Src.data import load_ohlcv_panel, load_symbol_map
from Src.backtest.cliques import CliqueTimeline
from Src.backtest.config import BacktestConfig, BasketParams, split_calibration_backtest
from Src.backtest.engine import BacktestEngine
from Src.backtest import metrics as bt_metrics
from Src.backtest import plots as bt_plots
from Src.backtest import runs as bt_runs

print("Imports OK")

## 1. Run-Konfiguration

Der einzige Ort, den ein von diesem Template abgeleitetes Notebook üblicherweise ändert. `STRATEGIES` listet alle zu vergleichenden Strategie-Namen -- jeder läuft mit denselben `BASKETS`/`BACKTEST_RANGE`/`INITIAL_CAPITAL`, aber eigenen `BasketParams` (siehe `STRATEGY_PARAMS`), da `garch_wilks` andere Stellschrauben braucht als `pairwise_zscore` (Wilks-/GARCH-Parameter statt `entry_z`/`exit_z`).

In [ ]:
RUN_LABEL_PREFIX = "TEMPLATE_DO_NOT_RUN"  # beim Kopieren AENDERN, z.B. "strategy_compare_basket1"
DESCRIPTION = ""

DATA_START = pd.Timestamp("2024-07-23", tz="UTC")   # fruehestmoegliches Datum im OHLCV-Panel
DATA_END = pd.Timestamp("2026-07-22", tz="UTC")     # spaetestmoegliches Datum
BACKTEST_FRACTION = 0.2  # gleiches Muster wie _template.ipynb / 01_forward_return_screening.ipynb

BASKETS = ("Basket 1",)
CLIQUE_VARIANT = "raw_24h"  # raw|residual x 6h|12h|24h, siehe data/processed/cliques/

STRATEGIES = ("pairwise_zscore", "garch_wilks")  # Reihenfolge = Lauf-Reihenfolge

INITIAL_CAPITAL = 100_000.0

In [ ]:
# Gemeinsame Basis-Parameter (Formation/Kosten/Positionslimits) -- pro Strategie
# überschrieben um die jeweils spezifischen Signalparameter (siehe Src/backtest/config.py:
# entry_z/exit_z/stop_z fuer pairwise_zscore, wilks_*/garch_* fuer garch_wilks).
COMMON_PARAMS = dict(
    formation_window_candles=288 * 7,
    max_candidate_pairs=3, pair_prefilter_top_n=15,
    min_half_life_candles=6, max_half_life_candles=288 * 2,
    max_concurrent_positions=3,
    fee_bps=10.0, slippage_bps=2.0,
)

STRATEGY_PARAMS: dict[str, BasketParams] = {
    "pairwise_zscore": BasketParams(entry_z=2.0, exit_z=0.5, stop_z=4.0, **COMMON_PARAMS),
    "garch_wilks": BasketParams(
        wilks_lookback_candles=288, wilks_p_content=0.95, wilks_confidence=0.95,
        garch_dist="t", garch_max_iter=500, stop_z=None, **COMMON_PARAMS,
    ),
}

# Pro-Basket-Overrides gelten fuer ALLE Strategien gleichermassen -- z.B. Basket 4
# (illiquide) braucht andere Kosten unabhaengig von der Signal-Logik:
BASKET_OVERRIDES: dict[str, dict] = {
    # "Basket 4": dict(fee_bps=10.0, slippage_model="high_low", capital_per_position=0.05),
}

## 2. Daten laden

Einmal für alle Strategien gemeinsam geladen -- dieselben Preise/Cliquen, damit der Vergleich fair ist (kein Unterschied durch unterschiedliche Datenausschnitte).

In [ ]:
id_to_symbol, symbol_to_id = load_symbol_map()

# Panel muss VOR DATA_START genug Historie fuer formation_window_candles haben
# (Warm-up-Guard G7) -- hier grosszuegig ab DATA_START selbst geladen.
panel = load_ohlcv_panel(start=DATA_START, end=DATA_END, fields=("open", "high", "low", "close"))
print(f"Panel: {panel['close'].shape[0]:,} Kerzen x {panel['close'].shape[1]} Symbole")

In [ ]:
timeline = CliqueTimeline.from_cache(
    variant=CLIQUE_VARIANT, id_to_symbol=id_to_symbol, baskets=list(BASKETS),
    start=DATA_START, end=DATA_END,
)
print(timeline.describe())

## 3. Calibration/Backtest-Split

Derselbe Split wie in jedem anderen `04_backtest`-Notebook -- **alle** Strategien laufen auf demselben `BACKTEST_RANGE`, nie auf `CALIBRATION_RANGE` (siehe `split_calibration_backtest`).

In [ ]:
CALIBRATION_RANGE, BACKTEST_RANGE = split_calibration_backtest(DATA_START, DATA_END, fraction=BACKTEST_FRACTION)
assert CALIBRATION_RANGE.end < BACKTEST_RANGE.start  # keine Ueberlappung, hart geprueft
print(f"Calibration: {CALIBRATION_RANGE.start.date()} -> {CALIBRATION_RANGE.end.date()}")
print(f"Backtest:    {BACKTEST_RANGE.start.date()} -> {BACKTEST_RANGE.end.date()}")

if RUN_LABEL_PREFIX == "TEMPLATE_DO_NOT_RUN":
    raise SystemExit(
        "Dies ist die Kopiervorlage -- bitte zu einem neuen Notebook kopieren "
        "(cp 06_strategy_comparison.ipynb NN_kurzname.ipynb) und RUN_LABEL_PREFIX in "
        "Abschnitt 1 aendern, bevor der Vergleich ausgefuehrt wird."
    )

## 4. Jede Strategie ausführen & speichern

EIN `BacktestEngine`-Lauf pro Strategie, gleiche Baskets/Zeitraum/Kapital -- jeder Run wird sofort via `bt_runs.save_run` gespeichert (analog zu jedem anderen `04_backtest`-Notebook), `run_ids` sammelt die IDs für den Vergleich in Abschnitt 5.

In [ ]:
results: dict[str, "BacktestResult"] = {}
run_ids: dict[str, str] = {}

BENCHMARK_SYMBOL = "BTCUSDT"
benchmark_close = panel["close"][BENCHMARK_SYMBOL] if BENCHMARK_SYMBOL in panel["close"].columns else None

for strategy_name in STRATEGIES:
    print(f"=== {strategy_name} ===")
    config = BacktestConfig(
        label=f"{RUN_LABEL_PREFIX}-{strategy_name}", description=DESCRIPTION,
        calibration_range=CALIBRATION_RANGE, backtest_range=BACKTEST_RANGE,
        baskets=BASKETS, clique_variant=CLIQUE_VARIANT, strategy_name=strategy_name,
        default_params=STRATEGY_PARAMS[strategy_name], basket_overrides=BASKET_OVERRIDES,
        initial_capital=INITIAL_CAPITAL,
    )
    engine = BacktestEngine(config=config, panel=panel, timeline=timeline)
    print(engine.describe_guards())

    t0 = time.time()
    result = engine.run(progress=True)
    runtime_seconds = time.time() - t0
    print(f"Laufzeit: {runtime_seconds:.1f}s | {len(result.trades)} Trades | Warnungen: {len(engine.warnings)}")

    result.metrics = bt_metrics.compute_all_metrics(
        equity_curve=result.equity_curve, trades=result.trades,
        benchmark_close=benchmark_close, config=config,
    )
    result.metrics_by_basket = bt_metrics.metrics_by_basket(result.trades, config)
    result.meta["runtime_seconds"] = runtime_seconds

    run_dir = bt_runs.save_run(result)
    print(f"Gespeichert: {result.run_id}\n{run_dir}\n")

    results[strategy_name] = result
    run_ids[strategy_name] = result.run_id

print("Alle Strategien durchgelaufen:", run_ids)

## 5. Kennzahlen-Vergleich

Selbe Bausteine wie `04_run_comparison.ipynb` (`bt_runs.compare_runs`/`diff_configs`/`compare_equity_curves`/`compare_trade_distributions`), hier direkt auf den soeben erzeugten `run_ids` statt manuell eingetragener IDs.

In [ ]:
display(bt_runs.compare_runs(list(run_ids.values())))

## 6. Config-Unterschiede

Zeigt, worin sich die Strategien tatsächlich unterscheiden (Signalparameter) -- Formation/Kosten/Positionslimits sollten wegen `COMMON_PARAMS` identisch bleiben.

In [ ]:
display(bt_runs.diff_configs(list(run_ids.values())))

## 7. Equity-Kurven (normiert)

In [ ]:
curves = bt_runs.compare_equity_curves(list(run_ids.values()))
curves.plot(figsize=(11, 5), title="Equity-Kurven im Vergleich (normiert auf 1.0 bei Backtest-Start)")

## 8. Trade-Verteilung nach Exit-Grund

`pairwise_zscore` kennt `target`/`stop`/`timeout`/`clique_break`/`data_gap` (siehe `src/backtest/exit_rules.py`); `garch_wilks` nutzt aktuell nur `target` als strategiespezifischen Exit (Wilks-Exit-Mitte) plus dieselben Engine-weiten Break-Regeln.

In [ ]:
display(bt_runs.compare_trade_distributions(list(run_ids.values())))

## 9. Weitere gespeicherte Runs

Falls frühere Läufe (auch aus anderen Notebooks) mit in den Vergleich sollen -- `bt_runs.list_runs()` zeigt alle, IDs lassen sich in Abschnitt 5-8 manuell ergänzen.

In [ ]:
display(bt_runs.list_runs(refresh_index=True).tail(10))

## 10. Interpretation

_Freitext: welche Strategie schneidet auf diesem Zeitraum/Basket besser ab, und an welcher Kennzahl (Sharpe? Max-Drawdown? Anzahl Trades?) zeigt sich das am deutlichsten? Sind die Exit-Verteilungen plausibel (z.B. dominiert bei `garch_wilks` `target`, weil die Wilks-Exit-Mitte im Gegensatz zu einem festen `exit_z` sehr reaktionsschnell ist)? Ist der Unterschied robust über mehrere Baskets/Zeiträume, oder nur ein Artefakt dieses einen Laufs?_